faz a conversão de tipos (datas, quantidade de passageiros para inteiro e ganhos para decimal/double) e já cria colunas agregadas úteis, como o total de passageiros do dia e o ganho total (manhã + noite):

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

catalogo = "will"
esquema = "i30"

tabela_bronze = f"{catalogo}.{esquema}.bronze_caronas"
tabela_silver = f"{catalogo}.{esquema}.silver_caronas"

# 1. Leitura dos dados da Bronze
df_bronze = spark.table(tabela_bronze)

# 2. Transformações e padronização para a Camada Silver
df_silver = (
    df_bronze
    .withColumn("DATA", F.to_date(F.col("DATA"), "dd/MM/yyyy"))
    # Mapeia os nomes reais da Bronze para a Silver
    .withColumn("PASSAG_MANHA", F.col("MANHA_QTD").cast(IntegerType()))
    .withColumn("GANHO_MANHA_REAIS", F.col("MANHA_RS").cast(DoubleType()))
    .withColumn("PASSAG_NOITE", F.col("NOITE_QTD").cast(IntegerType()))
    .withColumn("GANHO_NOITE_REAIS", F.col("NOITE_RS").cast(DoubleType()))
    # Enriquecimento: Métricas totais do dia
    .withColumn("TOTAL_PASSAGEIROS", F.coalesce(F.col("PASSAG_MANHA"), F.lit(0)) + F.coalesce(F.col("PASSAG_NOITE"), F.lit(0)))
    .withColumn("TOTAL_GANHO_REAIS", F.coalesce(F.col("GANHO_MANHA_REAIS"), F.lit(0.0)) + F.coalesce(F.col("GANHO_NOITE_REAIS"), F.lit(0.0)))
    # Seleção final limpa
    .select(
        "DATA",
        "PASSAG_MANHA",
        "GANHO_MANHA_REAIS",
        "PASSAG_NOITE",
        "GANHO_NOITE_REAIS",
        "TOTAL_PASSAGEIROS",
        "TOTAL_GANHO_REAIS",
        "_source_file"
    )
    .withColumn("_processed_at", F.current_timestamp())
)

# 3. Gravação na Camada Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)

print(f"Camada Silver de caronas gravada com sucesso: {tabela_silver}")
display(spark.table(tabela_silver))